# Import Required Libraries
Import necessary libraries such as pandas, numpy, matplotlib, and any file handling libraries needed to read log files.

In [1]:
# Import necessary libraries
import pandas as pd  # For data manipulation and analysis
import numpy as np  # For numerical operations
import matplotlib.pyplot as plt  # For data visualization
import os  # For file handling and directory operations
from pathlib import Path
import re
import asyncio
import sys
from datetime import datetime

Path.cwd()

ROOT = Path.cwd().parents[0]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from app.config.config import MODEL_NAME, MODEL_TEMPERATURE

# Load Individual Log Analysis Files
Load all individual log analysis files from a specified directory using appropriate file reading methods (CSV, JSON, or other formats).

In [2]:
output_dir = Path(ROOT, "app/logs/analysis_results_LaaJ")
output_dir.mkdir(parents=True, exist_ok=True)
batch_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
print(f"Current timestamp: {batch_timestamp}")

# Prepare analysis prompt
ANALYSIS_MODEL = "gemini-3-flash-preview:cloud"  # Can override with specific model
ANALYSIS_TEMP = 0.0  # Lower temperature for more focused analysis
MAX_ROWS = None  # Set to None to analyze all rows, or specify a number to limit (e.g., 50)
SAMPLE_STRATEGY = 'all'  # 'all', 'evenly_spaced', or 'head'


Current timestamp: 20260215_075325


# Utils

In [3]:
async def call_llm(instruction: str, user_text: str, model_name: str, temperature: float = 0.2) -> str:
    """Call Ollama API for LLM-based analysis."""
    from app.src.ollama_api import call_ollama

    prompt = f"{instruction.strip()}\n\nUser Input:\n{user_text.strip()}\n"
    resp = await call_ollama(
        prompt, model_name, use_stream=True, temperature=temperature
    )
    if not resp or not str(resp).strip():
        raise RuntimeError("LLM returned empty response")
    return str(resp).strip()

In [4]:
import re
from pathlib import Path

def parse_filename(file_name):
    """
    Extract metadata from filename format:
    individual_orpda_20260206_171410_gemini-3-flash-preview-cloud_0.8_hailey_20260213_101643.txt
    or
    individual_orpa_20260207_111903_cogito-2.1-671b-cloud_0.0_hailey_20260213_101643.txt
    """
    try:
        # Remove extension
        name_without_ext = file_name.replace('.txt', '')
        
        # Use regex to parse: individual_MODE_DATE_TIME_MODEL_TEMP_AGENT_TIMESTAMP
        # Temperature can be 0.0, 0.5, 1.0, etc.
        pattern = r'individual_(\w+)_(\d{8})_(\d{6})_(.+?)_([\d\.]+)_(\w+)_(\d{8}_\d{6})'
        match = re.match(pattern, name_without_ext)
        
        if match:
            mode, session_date, session_time, model, temperature, agent, analysis_timestamp = match.groups()
            
            return {
                'mode': mode.upper(),
                'session_date': session_date,
                'session_time': session_time,
                'model': model,
                'temperature': float(temperature),
                'agent': agent.capitalize(),
                'analysis_timestamp': analysis_timestamp,
                'filename': file_name
            }
        else:
            print(f"Filename does not match expected pattern: {file_name}")
            return None
            
    except Exception as e:
        print(f"Error parsing {file_name}: {e}")
        return None

# Test with sample filenames
samples = [
    "individual_orpda_20260206_171410_gemini-3-flash-preview-cloud_0.8_hailey_20260213_101643.txt",
    "individual_orpda_20260210_142355_gpt-oss-20b-cloud_1.0_sam_20260213_101643.txt",
    "individual_orpa_20260207_111903_cogito-2.1-671b-cloud_0.0_hailey_20260213_101643.txt"
]

for sample in samples:
    parsed = parse_filename(sample)
    if parsed:
        print(f"✓ {parsed['agent']} | {parsed['model']} | Temp: {parsed['temperature']}")

✓ Hailey | gemini-3-flash-preview-cloud | Temp: 0.8
✓ Sam | gpt-oss-20b-cloud | Temp: 1.0
✓ Hailey | cogito-2.1-671b-cloud | Temp: 0.0


In [5]:
import os
import pandas as pd

# Specify the directory containing the log analysis files
log_dir = "../app/logs/analysis_results_LaaJ/v7"

# Initialize an empty list to store parsed metadata and file content
log_data = []

# Iterate through all files in the specified directory
for file_name in os.listdir(log_dir):
    # Check if file starts with "individual_" and ends with ".txt"
    if file_name.startswith("individual_") and file_name.endswith(".md"):
        file_path = os.path.join(log_dir, file_name)
        
        # Parse metadata from filename
        metadata = parse_filename(file_name)
        
        if metadata:
            try:
                # Load file content
                with open(file_path, 'r', encoding='utf-8') as f:
                    content = f.read()
                
                # Add content to metadata
                metadata['content'] = content
                metadata['path'] = file_path
                
                log_data.append(metadata)
                
            except Exception as e:
                print(f"Error loading {file_name}: {e}")

# Convert to DataFrame for easier filtering and analysis
df_logs = pd.DataFrame(log_data)

print(f"Loaded {len(df_logs)} individual analysis files from {log_dir}\n")
print(df_logs[['filename', 'mode', 'model', 'temperature', 'agent']].head(10))
print(f"\nShape: {df_logs.shape}")
print(f"\nUnique agents: {df_logs['agent'].unique()}")
print(f"Unique models: {df_logs['model'].unique()}")
print(f"Temperature range: {df_logs['temperature'].min()} - {df_logs['temperature'].max()}")

Loaded 47 individual analysis files from ../app/logs/analysis_results_LaaJ/v7

                                            filename   mode  \
0  individual_orpda_20260213_171058_gemini-3-flas...  ORPDA   
1  individual_orpa_20260214_073056_gemini-3-flash...   ORPA   
2  individual_orpa_20260214_073103_gemini-3-flash...   ORPA   
3  individual_orpda_20260214_072656_gemini-3-flas...  ORPDA   
4  individual_orpda_20260213_190432_gemini-3-flas...  ORPDA   
5  individual_orpda_20260214_073041_gemini-3-flas...  ORPDA   
6  individual_orpa_20260214_110628_gemini-3-flash...   ORPA   
7  individual_orpda_20260213_174840_gemini-3-flas...  ORPDA   
8  individual_orpda_20260213_200257_gemini-3-flas...  ORPDA   
9  individual_orpa_20260214_110845_gemini-3-flash...   ORPA   

                          model  temperature     agent  
0  gemini-3-flash-preview-cloud          0.5     Maria  
1  gemini-3-flash-preview-cloud          0.3       Sam  
2  gemini-3-flash-preview-cloud          0.5       Sam  

In [6]:
df_logs.model.unique()

array(['gemini-3-flash-preview-cloud'], dtype=object)

# Apply Filters to Log Data
Apply the defined filters to the loaded log data using pandas filtering operations to keep only records that match the specified criteria.

In [7]:
models = df_logs.model.unique()
print(models)

['gemini-3-flash-preview-cloud']


In [21]:
# Define filter parameters in a dictionary for easy access
FILTER_PARAMS = {
    "temp_range": (0.3, 0.7),
    # "agent": "Maria",
    "model": models[0],
    "mode": "ORPDA",
}

# Apply Filters to Log Data
filtered_logs = df_logs.copy()


# Apply temperature filter
if 'temp_range' in FILTER_PARAMS.keys():
    filtered_logs = filtered_logs[
        (df_logs['temperature'] >= FILTER_PARAMS['temp_range'][0]) &
        (df_logs['temperature'] <= FILTER_PARAMS['temp_range'][1])
    ]

# Apply persona (agent name) filter
if 'agent' in FILTER_PARAMS.keys():
    filtered_logs = filtered_logs[
        filtered_logs['agent'] == FILTER_PARAMS['agent']
    ]

# Apply model name filter
if 'model' in FILTER_PARAMS.keys():
    filtered_logs = filtered_logs[
        filtered_logs['model'] == FILTER_PARAMS['model']
    ]
    
# Apply mode name filter
if 'mode' in FILTER_PARAMS.keys():
    filtered_logs = filtered_logs[
        filtered_logs['mode'] == FILTER_PARAMS['mode']
    ]

# Display the first few rows of the filtered dataframe
filtered_logs

,mode,session_date,session_time,model,temperature,agent,analysis_timestamp,filename,content,path
0,ORPDA,20260213,171058,gemini-3-flash-preview-cloud,0.5,Maria,20260215_005946,individual_orpda_20260213_171058_gemini-3-flas...,Analysis of: cleaned_session_orpda_20260213_17...,../app/logs/analysis_results_LaaJ/v7/individua...
3,ORPDA,20260214,072656,gemini-3-flash-preview-cloud,0.3,Hailey,20260215_005946,individual_orpda_20260214_072656_gemini-3-flas...,Analysis of: cleaned_session_orpda_20260214_07...,../app/logs/analysis_results_LaaJ/v7/individua...
4,ORPDA,20260213,190432,gemini-3-flash-preview-cloud,0.7,Isabella,20260215_005946,individual_orpda_20260213_190432_gemini-3-flas...,Analysis of: cleaned_session_orpda_20260213_19...,../app/logs/analysis_results_LaaJ/v7/individua...
5,ORPDA,20260214,073041,gemini-3-flash-preview-cloud,0.3,Sam,20260215_005946,individual_orpda_20260214_073041_gemini-3-flas...,Analysis of: cleaned_session_orpda_20260214_07...,../app/logs/analysis_results_LaaJ/v7/individua...
7,ORPDA,20260213,174840,gemini-3-flash-preview-cloud,0.7,Maria,20260215_005946,individual_orpda_20260213_174840_gemini-3-flas...,Analysis of: cleaned_session_orpda_20260213_17...,../app/logs/analysis_results_LaaJ/v7/individua...
8,ORPDA,20260213,200257,gemini-3-flash-preview-cloud,0.3,Sam,20260215_005946,individual_orpda_20260213_200257_gemini-3-flas...,Analysis of: cleaned_session_orpda_20260213_20...,../app/logs/analysis_results_LaaJ/v7/individua...
10,ORPDA,20260213,194410,gemini-3-flash-preview-cloud,0.5,Hailey,20260215_005946,individual_orpda_20260213_194410_gemini-3-flas...,Analysis of: cleaned_session_orpda_20260213_19...,../app/logs/analysis_results_LaaJ/v7/individua...
11,ORPDA,20260213,194721,gemini-3-flash-preview-cloud,0.5,Isabella,20260215_005946,individual_orpda_20260213_194721_gemini-3-flas...,Analysis of: cleaned_session_orpda_20260213_19...,../app/logs/analysis_results_LaaJ/v7/individua...
12,ORPDA,20260214,174403,gemini-3-flash-preview-cloud,0.7,Maria,20260215_005946,individual_orpda_20260214_174403_gemini-3-flas...,Analysis of: cleaned_session_orpda_20260214_17...,../app/logs/analysis_results_LaaJ/v7/individua...
13,ORPDA,20260214,072810,gemini-3-flash-preview-cloud,0.5,Isabella,20260215_005946,individual_orpda_20260214_072810_gemini-3-flas...,Analysis of: cleaned_session_orpda_20260214_07...,../app/logs/analysis_results_LaaJ/v7/individua...


In [22]:
batch_results = []

for i, x in filtered_logs.iterrows():
    batch_results.append({
        'log': x['filename'],
        'agent': x['agent'],
        'model': x['model'],
        'mode': x['mode'],
        'temp': x['temperature'],
        'analysis': x
    })
    
from pprint import pprint
print(batch_results[-1]['analysis']['content'])

Analysis of: cleaned_session_orpda_20260214_174356_gemini-3-flash-preview-cloud_0.5_maria.csv
Agent: Maria Lopez
Model: gemini-3-flash-preview:cloud
Mode: ORPDA
Temperature: 0.5
Analyzed at: 20260215_005946
Analyzed by model: gemini-3-flash-preview:cloud
Analyzed by model temp: 0.0
Session: 46/47


This analysis evaluates the behavioral profile of Maria Lopez over a 57-action session (2023-02-13 10:00 to 2023-02-14 00:00) using the ORPDA architecture.

### 1. Layer Function Validation

**OBSERVATION LAYER**:
*   **Accuracy**: `state_summary_o` consistently captures the environmental context (e.g., transitioning from the sensory-rich bathroom to the quiet library).
*   **Sufficiency**: `environment_description_o` is excellent, providing the "behavioral affordances" (phone buzzing, scent of chalk, clinking silverware) that explain why the agent drifts.
*   **Consistency**: Perception remains stable. High-energy mornings are reflected in the observation of "bright morning light," while ev

# Generate Global Analysis Report
Aggregate and analyze the filtered data to generate summary statistics, metrics, and insights across all filtered log entries.

In [23]:
# Run global comparative analysis across all batch results
if 'batch_results' in locals() and len(batch_results) > 1:
    # print(f"\n{'='*80}")
    # print(f"GLOBAL COMPARATIVE ANALYSIS ({len(batch_results)} sessions)")
    # print(f"{'='*80}\n")
    
    # Prepare comprehensive global analysis prompt
    global_instruction = """You are an expert AI behavior analyst and cognitive neuroscientist conducting a systematic comparative study across multiple agent sessions.

Analyze the following collection of agent sessions and provide a COMPREHENSIVE GLOBAL COMPARATIVE ANALYSIS focused on:

## PART 1: LAYER-BY-LAYER FUNCTIONAL CONSISTENCY (ORPDA Architecture)

### 1.1 OBSERVATION LAYER (Cross-Session Analysis):
- **Perception Consistency**: Do all models perceive the same situations consistently?
- **Perceptual Biases**: Are there systematic perceptual biases or selective attention patterns?
- **Environmental Context Capture**: Are `environment_description_o` details sufficient across models/temperatures?
- **Model Comparison**: Which models show most accurate/complete observations?

### 1.2 REFLECTION LAYER (Meta-rule Control & Executive Function):
- **Meta-rule Function**: Is `meta_rule_r` (continue vs reset_plan) functioning as executive control?
  * What triggers transition from "continue" → "reset_plan"?
  * Can the agent exit "reset_plan" back to "continue"? (Trap detection)
  * Is this pattern consistent across models/temperatures?
- **Metacognitive Insight**: Does `reasoning_r` show genuine metacognitive monitoring?
  * Pattern recognition quality vs repetitive categorization
  * Error detection (when does reflection recognize failure)?
  * Causality attribution (does it explain *why* drift occurred)?
- **State Reflection Accuracy**: Does `state_summary_r` at time t correctly reflect `state_summary_a` at time t-1?
  * Temporal alignment: Is the reflection actually processing prior actions?
  * Conceptual alignment: Does reflected understanding match actual prior state?
- **Thought Pattern Evolution**: Does `emerging_thought_pattern_r` show meaningful progression or stasis?
- **Model-Temperature Effects**: 
  * Does higher temperature improve/degrade reflection quality?
  * Do certain models show more robust reflection?
  * Neuroscience alignment: PFC-like metacognitive performance

### 1.3 PLAN LAYER (Goal-Directed Behavior & Forward Modeling):
- **Plan Adaptation**: Does plan change after reset_plan, or does it repeat?
  * When reflection identifies drift, does plan incorporate new strategies?
  * Evidence of learning vs. stuck patterns?
- **Realistic Goal Structure**: Are plans hierarchically organized (abstract → concrete)?
  * Simple action sequences vs goal hierarchies
  * Integration of competing motivations (focus vs rewards)?
- **Forward Modeling**: Does plan show evidence of outcome prediction?
  * Proactive adjustments (planning to resist temptations)?
  * Consideration of environmental affordances?
- **Context Integration**: Does `state_summary_p` incorporate observation context?
- **Model Comparison**: Which models create more sophisticated/realistic plans?
- **Neuroscience Grounding**: Orbitofrontal cortex (OFC) / medial prefrontal cortex function?

### 1.4 DRIFT LAYER (Behavioral Inhibition & Competing Goals) [ORPDA only]:
- **Drift Detection Appropriateness**:
  * When `should_drift_d` = True, what triggered it? (task difficulty, reward salience, time pressure?)
  * When `should_drift_d` = False, is drift truly absent or masked?
- **Drift Layer Power Balance**:
  * **Dominant Drift**: Does drift layer override plan layer too frequently?
  * **Weak Drift**: Is drift layer properly detecting actual behavioral failures?
  * **Optimal Balance**: When does drift control feel realistic?
- **Explicit Drift Typology**: Are drift types appropriately classified?
  * Behavioral drift (overt action change)
  * Internal drift (covert attention/thought change)
  * Reward-seeking drift (specific motivational hijacking)
- **Explicit vs Implicit Drift Alignment**:
  * When `should_drift_d` = True, does content (state_summary_a, drift_topic_a) show actual drift? (Agreement)
  * When `should_drift_d` = False, does implicit analysis reveal hidden drift? (Leaky inhibition)
  * Frequency of explicit-implicit agreement/disagreement
- **Recovery Strategies**: Are `potential_recovery_d` suggestions realistic and evidence-based?
- **Neuroscience Grounding**: Anterior cingulate cortex (ACC) drift detection, dorsolateral prefrontal cortex (dlPFC) inhibition

### 1.5 ACTION LAYER (Motor Execution & Behavioral Outcome):
- **Plan-Action Coupling**:
  * Explicit alignment: `action_a` matches `action_p` label (%)
  * When misalignment occurs, is it due to Plan change or Drift override?
- **State Summary Fidelity**: Does `state_summary_a` accurately describe actual behavior?
  * Detailed vs generic descriptions
  * Emotional/cognitive content included?
- **Drift Integration**: When Plan and Drift conflict, which dominates?
  * Probabilistic resolution (realistic) vs deterministic (unrealistic)?
  * Does outcome match realistic behavioral inhibition failures?
- **Action Execution Realism**: Are behavioral changes instantaneous or gradual?
- **Location-Action Coherence**: Does `action_a` align with `location_a`?
- **Neuroscience Grounding**: Motor cortex/basal ganglia action execution?

### 1.6 Cross-Layer Information Flow:
- **Layer Utilization**: Do all layers contribute meaningfully, or are some outputs ignored?
- **Information Cascade**: Observation → Reflection → Plan → [Drift] → Action (tested)
- **Constraint Propagation**: Do earlier layers appropriately constrain later layers?
- **Layer Contradiction Detection**: Cases where layers conflict (e.g., reflection detects drift, plan doesn't adapt)
- **Model Differences**: Which models show tighter cross-layer integration?

---

## PART 2: PLAN-ACTION ALIGNMENT (Explicit + Implicit)

### 2.1 Explicit Alignment (Label-Level):
- **Action Label Alignment Rate**: %matched across all sessions
  * Per-model comparison
  * Temperature effects
  * Mode effects (ORPA vs ORPDA)
- **Location Label Alignment Rate**: %matched across all sessions
- **Mismatch Patterns**:
  * When do misalignments occur? (specific times, activities, transitions?)
  * Frequency by model/temperature
  * Systematic vs random patterns?

### 2.2 Implicit Alignment (Content-Level / Semantic Drift):
- **State Summary Semantic Alignment**: Compare `state_summary_p` vs `state_summary_a`
  * Word overlap / semantic similarity metrics
  * Thematic coherence
- **Performing vs Executing Gap Detection**:
  * Cases where labels match but content reveals actual drift
  * Frequency by model/temperature
  * Examples: "action_a = study" but "state_summary_a = studying while distracted by phone"
- **Linguistic Indicators of Misalignment**:
  * Confidence markers (certain vs uncertain language)
  * Tense and agency (active vs passive voice)
  * Emotional/motivational content divergence

### 2.3 Explicit vs Implicit Agreement:
- **Perfect Alignment** (both explicit and implicit high): Conditions enabling true behavioral alignment
- **High Explicit, Low Implicit** (label match, content drift): "Performing vs Executing" gaps
  * Frequency and severity by model
  * Cognitive load indicators
- **Low Explicit, High Implicit** (label mismatch, content coherent): When semantic coherence survives label change
- **Leaky Inhibition Analysis**:
  * Implicit drift despite explicit inhibition (meta_rule = focus, but state_summary shows distraction)
  * Relationship to temperature/model
  * Frequency rates

---

## PART 3: DRIFT PATTERN ANALYSIS (Explicit + Implicit)

### 3.1 Explicit Drift (ORPDA Mode Only):
- **Drift Frequency**: Average drifts per session
  * Per-model comparison
  * Distribution across session timeline
- **Drift Triggers**: What causes explicit drift detection?
  * Task difficulty / cognitive load
  * Environmental salience / reward availability
  * Time-of-day effects
- **Meta-rule Relationship**: When `should_drift_d` = True, what does `meta_rule_r` show?
  * Correlation between drift detection and reset_plan transitions
  * Does meta_rule respond appropriately to drift?
- **Drift Type Distribution**: behavioral vs internal vs reward-seeking
  * Per-model preferences
  * Temporal patterns

### 3.2 Implicit Drift (All Modes):
- **Content-Level Topic Divergence**: Analyze `state_summary_a`, `drift_topic_a`, `drift_action_d` for semantic drift from plan
  * Word divergence from `state_summary_p` and `topic_a`
  * Thematic shift detection
- **Implicit Drift in ORPA Mode**: Since ORPA lacks explicit drift layer, measure only implicit drift
  * How frequent is implicit drift without explicit marker?
  * Does content show topical divergence even though `should_drift_d` column absent?
- **Linguistic Variability**:
  * Sentence-level vocabulary changes (micro-stochastic drift)
  * Topic-level thematic shifts (macro-stochastic drift)
- **Model-Temperature Effects**:
  * Higher temperature → more implicit drift? (linguistic variability)
  * Certain models → more robust topical coherence?

### 3.3 Explicit vs Implicit Drift Comparison:
- **Agreement Rate**: When explicit drift = True, is there implicit content drift?
  * Perfect agreement: explicit marker matches content divergence
  * Disagreement: explicit drift without content change (false positive detection?)
- **Leaky Inhibition Patterns**:
  * Implicit drift when explicit inhibition active (`should_drift_d` = False)
  * Frequency by model/temperature
  * Indicates inhibitory control failures (realistic)
- **Drift Detection Quality**:
  * Sensitivity: Does explicit drift detect all actual implicit drift?
  * Specificity: Does explicit drift avoid false positives?
  * Per-model performance
- **ORPA vs ORPDA Comparison**:
  * ORPA: Only implicit drift, no explicit detection mechanism
  * ORPDA: Both explicit and implicit drift measured
  * Which mode better predicts actual behavioral drift?

### 3.4 Drift Variability & Diversity:
- **Topic Variability** (both explicit and implicit):
  * How diverse are drifted topics across session?
  * Repetitive drift (same topic) vs varied drift
  * Per-model comparison (which models show most diverse drifts?)
- **Linguistic Variability** (micro-stochastic):
  * Vocabulary diversity in describing similar actions
  * Paraphrase diversity
  * Temperature effects on micro-stochasticity
- **Semantic Coherence of Drift**:
  * Do drifted topics semantically cohere (e.g., all entertainment-related) or random?
  * Associative thinking patterns
  * Model-specific coherence quality

---

## PART 4: MODEL & TEMPERATURE COMPARISON

### 4.1 Model-to-Model Performance:
- **Layer Function Consistency**: Which models maintain consistent ORPDA layer behavior?
  * Reflection quality by model
  * Plan sophistication by model
  * Action fidelity by model
- **Alignment Rates**: Plan-action alignment (explicit and implicit) by model
  * Best vs worst performers
- **Drift Control**: Which models show best balance of drift detection?
  * Too permissive (too much drift)?
  * Too restrictive (unrealistic inhibition)?
  * Goldilocks balance?
- **Metacognitive Quality**: Reflection layer reasoning by model
  * Pattern recognition
  * Error attribution accuracy
- **Model Ranking**: 
  * Overall ORPDA architecture fit
  * Plan-action alignment quality
  * Drift control appropriateness
  * Cognitive realism

### 4.2 Temperature Effects:
- **ORPA Mode (Reflect-Plan-Act)**:
  * Micro-stochastic drift (sentence-level variability)
  * Higher temp → more sentence-level changes?
  * Impact on behavioral coherence?
- **ORPDA Mode (with Drift layer)**:
  * Macro-stochastic resilience (schema-level topical switching)
  * Does explicit drift mechanism override temperature effects?
  * Robust vs fragile architecture?
- **Layer-Specific Temperature Sensitivity**:
  * Reflection layer: Does higher temp reduce metacognitive quality?
  * Plan layer: Does higher temp create unrealistic plans?
  * Drift layer: Does higher temp affect drift detection appropriateness?

### 4.3 Architecture Group Comparison:
- **Open-Source vs Commercial Models**:
  * Performance comparison (both explicit and implicit drift rates)
  * Cost-benefit analysis for each mode
  * Which OSS models approach commercial performance?
- **Model Family Patterns** (if data available):
  * Gemini family performance
  * GPT family performance
  * Cogito/Gemma family performance
  * Family-specific strengths/weaknesses

### 4.4 Mode Comparison (ORPA vs ORPDA):
- **Drift Control Mechanisms**:
  * ORPA: Pure executive inhibition (Reflect layer only)
  * ORPDA: Biologically-constrained (Drift layer adds competing goals)
- **Behavioral Realism**:
  * Which mode produces more realistic inhibition failures?
  * Which mode better matches human behavioral patterns?
- **Explicit vs Implicit Drift**:
  * ORPA: Only implicit drift measurable
  * ORPDA: Both types measurable, can assess agreement
- **Cognitive Load Effects**:
  * Does ORPDA better model competing goal interference?
  * Does ORPA show unrealistic perfect inhibition?

---

## PART 5: COGNITIVE SCIENCE & NEUROSCIENCE GROUNDING

### 5.1 Task-Unrelated Thought (TUT) & Mind-Wandering:
- **TUT Detection**: Evidence of off-task thinking even when action stays on-task?
  * Implicit drift in state_summary while action_a remains planned
  * Frequency across sessions
- **Biological Plausibility**: Do observed patterns match neuroscience literature on TUT?
  * Default mode network (DMN) vs executive control network (ECN) competition?
  * Evidence of DMN-ECN interference patterns?

### 5.2 Executive Dysfunction Patterns:
- **ADHD-like patterns**: 
  * Sustained attention failures
  * Inhibitory control failures (leaky inhibition)
  * Impulsive drift to high-salience stimuli
- **OCD-like patterns**:
  * Perseverative behavior (stuck in reset_plan loop)
  * Ritualistic action sequences
- **Healthy baseline patterns**:
  * Evidence of adaptive inhibition
  * Flexible goal-switching
  * Realistic error recovery

### 5.3 Inhibitory Control (dlPFC/ACC Function):
- **Inhibition Effectiveness**: When should agents resist drift, do they?
  * Success rate by model/temperature
  * Conditions enabling successful inhibition
  * Signs of prefrontal fatigue/depletion
- **Inhibition Realism**: 
  * Perfect inhibition (unrealistic - suggests ORPA weakness)
  * Probabilistic inhibition (realistic - suggests ORPDA strength)
  * Fatigue effects over time

### 5.4 Temperature Effects on Behavior:
- **Stochasticity Types**:
  * **Micro-stochastic**: Sentence-level randomness, drunk person on yellow line (temperature increases)
  * **Macro-stochastic**: Schema switching, competing goal emergence (less temperature-dependent)
- **Behavioral Signature by Temperature**:
- **Optimal Temperature Range**: Which temperature produces most realistic behavior?

### 5.5 Hyper-fixation & Perseveration:
- **Perseverative Drift**: When agent drifts to same topic repeatedly
  * Frequency across sessions
  * Model-specific perseveration patterns
  * Sign of schema-level fixation vs micro-stochastic randomness
- **Hyper-focus Detection**: Obsessive focus on single topic despite plan changes
  * Positive indicator of deep engagement vs negative indicator of inflexible attention

### 5.6 Neuroscience Citation & Grounding:
- **Required Citations**: Properly cite neuroscience concepts with (Author, Year) and DOI
  * Task-unrelated thought
  * Executive function 
  * Inhibitory control
  * Default mode network 
- **Confidence Marking**: Explicitly state "citation unavailable" rather than fabricating references
- **Biological Plausibility**: Rank models/architectures by how well they simulate realistic neurocognitive processes

---

## PART 6: SUMMARY & RECOMMENDATIONS

### 6.1 Quantitative Comparison Table:
Create summary tables for:
- Layer function consistency scores (by model)
- Alignment rates (explicit and implicit, by model)
- Explicit vs implicit drift agreement (by model)
- Drift variability rankings (topic diversity, linguistic diversity)
- Temperature effects (per model, per layer)
- ORPA vs ORPDA performance comparison

### 6.2 Pattern Identification:
- **Consistent Patterns**: Behaviors replicated across models/temperatures
- **Variable Patterns**: Behaviors unique to specific models/conditions
- **Anomalies**: Unexpected or counterintuitive findings

### 6.3 Model Rankings:
1. **Overall ORPDA Architecture Fit**: Rank models by holistic performance
2. **Plan-Action Alignment**: Rank by explicit and implicit alignment quality
3. **Drift Control**: Rank by appropriateness of drift detection/execution
4. **Cognitive Realism**: Rank by alignment with neuroscience principles
5. **Metacognitive Quality**: Rank by reflection layer performance
6. **Drift Variability**: Rank by topic diversity in actual drifts (implicit analysis)
7. **Linguistic Coherence**: Rank by maintaining semantic consistency despite temperature
8. **Inhibitory Control**: Rank by realistic vs perfect inhibition

### 6.4 Architecture-Specific Recommendations:
- **Best for ORPA mode**: Which models excel without explicit drift layer?
- **Best for ORPDA mode**: Which models best balance explicit drift with realistic inhibition?
- **Best for Low Latency / Cost**: OSS models approaching commercial performance?
- **Best for Cognitive Realism**: Which models most closely match neuroscience predictions?

### 6.5 Temperature Optimization:
- **Optimal temperature for each model**: Balance realism with coherence
- **Mode-specific temperature recommendations**: ORPA vs ORPDA temperature effects

### 6.6 Research Implications:
- **Cognitive Science Insights**: What do these patterns tell us about human cognition?
- **AI Architecture Lessons**: How should AI systems be designed to better simulate realistic behavior?
- **Future Research Directions**: Outstanding questions revealed by this analysis

---

## OUTPUT REQUIREMENTS:

Provide:
1. **Quantitative Comparison Tables** (Excel-ready format)
2. **Cross-Layer Analysis** (comprehensive layer-by-layer functional assessment)
3. **Plan-Action Alignment Report** (explicit + implicit analysis with examples)
4. **Drift Pattern Summary** (explicit vs implicit drift comparison with model rankings)
5. **Model Performance Rankings** (8 dimensions as listed above)
6. **Neuroscience Grounding Section** (properly cited, confidence-marked)
7. **Recommendations** (specific, actionable, model/temperature/mode-specific)
8. **Anomalies & Open Questions** (findings that contradict expectations)
"""
    
    # Compile comprehensive summary of all sessions for global analysis
    global_summary = f"COMPREHENSIVE SESSION COLLECTION ({len(batch_results)} sessions)\n"
    global_summary += "="*80 + "\n\n"
    
    for i, result in enumerate(batch_results, 1):
        global_summary += f"\n{'─'*80}\n"
        global_summary += f"SESSION {i}: [{result['agent']}] {result['model']}\n"
        global_summary += f"Mode: {result['mode'].upper()} | Temperature: {result['temp']} | Rows: {result.get('rows', 'N/A')}\n"
        global_summary += f"{'─'*80}\n"
        global_summary += f"\nINDIVIDUAL ANALYSIS HIGHLIGHTS:\n"
        # Extract key sections from individual analysis
        analysis_lines = result['analysis']
        for j, line in enumerate(analysis_lines[:30]):  # First 30 lines
            global_summary += f"{line}\n"
        global_summary += "\n[...full individual analysis available...]\n"
        print(global_summary)
    
    global_summary += "\n" + "="*80 + "\n"
    global_summary += "REQUEST: Synthesize above analyses into comprehensive cross-layer, cross-model comparative report.\n"
    global_summary += "Focus on: Layer functionality, Plan-Action alignment (explicit + implicit), Drift patterns (explicit + implicit).\n"
    global_summary += "Include model rankings, temperature effects, neuroscience grounding, and specific recommendations.\n"
    
    try:
        print("Running comprehensive global comparative analysis...")
        print("(This may take 2-3 minutes for full cross-layer analysis)\n")
        
        global_analysis_result = await asyncio.wait_for(
            call_llm(global_instruction, global_summary, ANALYSIS_MODEL, ANALYSIS_TEMP),
            timeout=600  # Longer timeout for comprehensive analysis
        )
        
        print("\n" + "="*80)
        print("GLOBAL COMPARATIVE ANALYSIS - FINAL REPORT")
        print("="*80 + "\n")
        print(global_analysis_result)
        
        # Save global analysis to file
        global_analysis_path = Path(output_dir) / f"GLOBAL_COMPARATIVE_ANALYSIS_{batch_timestamp}_{result["model"]}.md"
        with open(global_analysis_path, 'w', encoding='utf-8') as f:
            f.write("="*80 + "\n\n")
            f.write(f"GLOBAL COMPARATIVE ANALYSIS\n")
            f.write(f"Analysis Date: {batch_timestamp}\n")
            f.write(f"Sessions Analyzed: {len(batch_results)}\n")
            f.write("="*80 + "\n\n")
            for filename in filtered_logs['filename'].to_list():
              f.write(filename + "\n")
            f.write("\n")
            f.write(global_analysis_result)
        
        print(f"\n✓ Global analysis saved to: {global_analysis_path}")
        
    except asyncio.TimeoutError:
        print("Global analysis timed out. Try reducing number of sessions or increasing timeout.")
        global_analysis_result = None
    except Exception as e:
        print(f"Error during global analysis: {e}")
        global_analysis_result = None

elif 'batch_results' in locals() and len(batch_results) == 1:
    print("⚠ Only one session analyzed. Global comparative analysis requires 2+ sessions for meaningful comparison.")
    global_analysis_result = None

else:
    print("⚠ No batch results available for global analysis.")
    global_analysis_result = None

COMPREHENSIVE SESSION COLLECTION (24 sessions)


────────────────────────────────────────────────────────────────────────────────
SESSION 1: [Maria] gemini-3-flash-preview-cloud
Mode: ORPDA | Temperature: 0.5 | Rows: N/A
────────────────────────────────────────────────────────────────────────────────

INDIVIDUAL ANALYSIS HIGHLIGHTS:
ORPDA
20260213
171058
gemini-3-flash-preview-cloud
0.5
Maria
20260215_005946
individual_orpda_20260213_171058_gemini-3-flash-preview-cloud_0.5_maria_20260215_005946.md
Analysis of: cleaned_session_orpda_20260213_171058_gemini-3-flash-preview-cloud_0.5_maria.csv
Agent: Maria Lopez
Model: gemini-3-flash-preview:cloud
Mode: ORPDA
Temperature: 0.5
Analyzed at: 20260215_005946
Analyzed by model: gemini-3-flash-preview:cloud
Analyzed by model temp: 0.0
Session: 1/47


This behavioral analysis is based on the provided 57-action session log for **Maria Lopez**.

### 1. Layer Function Validation (ORPDA Architecture)

**OBSERVATION LAYER**
*   **Accuracy**: `state_su